# pars 

In [ ]:
from pathlib import Path
import importlib
import tmspath_utils as tmsu
importlib.reload(tmsu)
date,start_time=tmsu.import_modules()

In [ ]:
detrendType="Windowed"

if detrendType=="Non-Windowed":
    typeOffsetRise="nowind_biexp"
    typeOffsetDecay="nowind_biexp"
elif detrendType=="Windowed":
    typeOffsetRise="wind_poly_lagrange"
    typeOffsetDecay="wind_poly_3"
elif detrendType=="ICA_Detrend":
    typeOffsetRise="ICA"
    typeOffsetDecay="ICA"
else:
    raise ValueError(
        "detrendType deve essere "
        "'Windowed', 'Non-Windowed' oppure 'ICA_Detrend'"
    )

json_data={
    # utils
    "date":date,
    "start_time":start_time,
    "do_filter_and_plot_raw":True,
    "showPlotsEnd":False,
    "r_sfreq":1024,
    "eeg_type":"tep",
    
    # filtering
    "l_freq":0.1,
    "h_freq":45,
    "powerline_freq":50,
    "broad_band_h_freq":250,

    # stimulation
    "do_pulseArtifactRej":True,
    "pulse_artifact_rej_timewindow_min":-0.002,
    "pulse_artifact_rej_timewindow_max":0.008,

    # epoching
    "do_prepare_epochs":True,    
    'epochs_timewindow_min': -0.400, # 0.15 ora più larga per poter calcolare Morlet e PCIst
    'epochs_timewindow_max': 0.400,
    'baseline_cor_tmin': -0.150,
    'baseline_cor_tmax': 0.00,
    "epochs_plot_timewindow_min":-0.100,
    "epochs_plot_timewindow_max":0.400,
    
    # channels and trials cleaning
    "do_clean_trials_channels":True,
    "do_chan_trials_selection_automatic":True,
    "bad_trials":[],
    "bad_channels":[],
   
    # detrend
    "trials_wise":True,
    "detrendExtremeTechinque":"max",
    "detrend_type":typeOffsetDecay,
    "do_detrend":typeOffsetDecay!="ICA",
    "do_detrend_onlyOffsetChans":False,
    "detrend_offsetStart":True,
    "detrend_lag_correction":False,
    "detrend_typeOffsetRise":typeOffsetRise,
    "detrend_typeOffsetDecay":typeOffsetDecay,
    "detrend_polOrder_preOffset":2,
    "detrend_slopeThr":0.5,
    "detrend_offsetCorrectionType":"Gaussian" if "nowind" not in typeOffsetDecay else False,
    "detrend_noise_seed": 42,
    "detrend_offsetOddSamples":5,
    "detrend_overall":True,
    "detrend_noWindowedOrder":1,
    "detrend_maxTimeWindowOffset":0.020,
    "detrend_minTimeWindowOffset":0.0,
    "detrend_fitConstraint":False if "nowind" not in typeOffsetDecay else True,

    # ICA
    "do_ica": True,
    "do_ica_continuum":False,
    "do_ica_manualCheck": False,
    "do_ica_eigThresh":0,
    "do_ica_automaticRej":False if detrendType=="ICA_Detrend" else True,
    "do_label_prob_threshold":0,
    "ICA_seed": 42,

    # standard features
    "do_standard_features":True,
    "do_fooof_features":False,
    
    # natural frequency  
    "do_tep_natural_frequency":True,
    "tep_nf_fmin":8.0,
    "tep_nf_fmax":30, #45.0 -> se 45Hz ci sono possibili power outliers vicino ai bordi in gamma, meglio tenere upper limit a 30Hz
    "tep_nf_baseline_window_ms":(-260,-10),  # 250 ms, pari a 2 cicli a 8 Hz -> ci consente di avere buona risoluzione estremo inf alpha
    "tep_nf_response_window_ms":(20,200),
        
    # PCIst
    "do_pcist":True,
    "pcist_baseline_window_ms":(-150,-10),
    "pcist_response_window_ms":(10,300),
    "pcist_k":1.2,
    "pcist_min_snr":1.1,
    "pcist_max_var":99,
    "pcist_embed":False,
    "pcist_n_steps":100,
    "pcist_baseline_corr":False,
    "pcist_plot_max_components":6,
    "pcist_safe_margin_ms":2.0,
    # PCIst baseline sensitivity -> utile per stimare effetto da baseline range su PCIst
    "pcist_baseline_sweep":True, # questo costa un po' a livello di tempo, per delle prove tieni pure in False, per analisi definitiva farei True
    "pcist_baseline_sweep_start_ms":-300,
    "pcist_baseline_sweep_end_ms":-10,
    "pcist_baseline_sweep_step_ms":10*5,
    "pcist_baseline_sweep_min_duration_ms":50,

    # synthetic data
    "use_synthetic_tep_ground_truth":False,
    "synthetic_tep_frequency_hz":20.0*0.5,
    "synthetic_tep_onset_sec":0.020,
    "synthetic_tep_tau_sec":0.180,
    "synthetic_tep_response_duration_sec":0.8,
    "synthetic_tep_amplitude_uv":12.0*5,
    "synthetic_tep_noise_uv":4.0,
    "synthetic_tep_frequency_jitter_hz":0.0,
    "synthetic_tep_amplitude_jitter":0.15,
    "synthetic_tep_phase_jitter_rad":0.0,
    "synthetic_tep_add_aperiodic_noise":True,
    "synthetic_tep_add_tms_artifact":False,
    "synthetic_tep_tms_artifact_uv":1000.0,
    "synthetic_tep_seed":42,

    # injected artifacts
    "do_artifact":False,
    "do_artifact_rise":0.005,
    "do_artifact_decay":0.1,
    "do_artifact_gain":-3e-6*30,
    "do_artifact_chans":["Cz"],

    # directory and subject
    "mainDir":r"C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data",
    "sourceData":"MAYER",
    "subject":"PP064",
    "dataType":"ASCII",
    "emispheric_stimulation":"SX",

}




# TMS-EEG Processing Pipeline

### 1. Preprocessing & Artifact Removal

* **Load EEG**
* **Apply montage**
* **Detect TMS events**
* **Remove TMS pulse artifact**
* **Broad-band filter** (0.1–250 Hz)
* **Notch filter** (50 / 100 / 150 / 200 / 250 Hz)

**Results folder:** `1.basic/`

---

### 2. Preliminary Cleaning

* **Create long temporary epochs** for visual or automatic cleaning
* **Automatic / manual trial and channel inspection**
* **Reject bad trials**
* **Mark bad channels** in `info["bads"]`

Bad channels are **not dropped** at this stage.

**Results folders:**

* `3.trials/preDetrend/`
* `1.basic/`
* `6.pkls/`

---

### 3. Epoching & Detrending

* **Create final epochs** using only retained trials
* **Preserve all EEG channels**
* **Propagate bad-channel labels** to final epochs
* **Downsample** to the processing rate, e.g. 1024 Hz
* **Average reference** using good channels only
* *Optional:* **Pre-detrend PSD / FOOOF**
* **Slope and offset analysis**
* **Apply the selected detrending strategy**

**Results folders:**

* `2.detrend/`
* `2.detrend/examples/`
* `3.trials/preDetrend/`
* `3.trials/postDetrend/`
* `7.FOOOF/preDetrend/`
* `6.pkls/`

---

### 4. ICA

* **Prepare ICA copy**

  * bad channels remain in the complete epoched object
  * ICA fitting uses only EEG channels not marked as bad

* **Fit ICA on good channels only**

* **Automatic / manual ICA component selection**

  * ICLabel criteria, if enabled
  * eigenvalue criteria, if enabled
  * manual component inspection, if enabled

* **Apply ICA correction** to the complete epoched object

**Results folder:**

* `4.postICA/<ICA_timestamp>/`

---

### 5. Finalizing

* **Final filter** (0.1–45 Hz)
* **Resample** to the original sampling rate
* **Interpolate bad channels**
* **Final average reference**
* **Save** `postICA_final`
* **Generate final diagnostic plots**
* **Calculate PKL and final-data hashes**

**Results folders:**

* `4.postICA/<ICA_timestamp>/`
* `6.pkls/`
* analysis root directory


---

### 6. Feature Extraction

* **Standard post-ICA features** in 5.Extra/FE/ and jsonfile
  * TEP energy
  * TEP integral
  * sample entropy
  * permutation entropy
  * aperiodic FOOOF parameters
  * seed-channel PLV
  * manually selected TEP peaks

  ```
* **PCIst** in 5.Extra/FE/PCIst/
  ```
  ```
* **TEP Natural Frequency** in 5.Extra/FE/NaturalFrequency/
  ```





# init

In [ ]:
json_data,experiment_dir,sub,fileName,savePath=tmsu.setup_tep_analysis(json_data)

# Inizializza l'analisi TEP per il soggetto corrente.
# La funzione:
# - legge e normalizza l'identificativo del soggetto;
# - legge e normalizza il lato di stimolazione;
# - verifica che emispheric_stimulation sia "SX" oppure "DX";
# - assegna automaticamente i seed channel in base all'emisfero:
#   - SX: AF3, F3, Fz, FC1
#   - DX: Fz, AF4, F4, FC2
# - costruisce la directory del soggetto a partire da mainDir;
# - costruisce il nome base dei file di input EEG;
# - rimuove un eventuale experiment_dir precedente;
# - richiama directorySetup() per:
#   - caricare nome, versione e data di rilascio della pipeline;
#   - generare un timestamp per l'analisi;
#   - creare la directory specifica dell'esperimento;
#   - creare tutte le sottocartelle della pipeline;
#   - salvare la configurazione iniziale nel file JSON del soggetto;
# - assegna l'identificativo dell'analisi;
# - registra il percorso base del file di input;
# - registra la directory di output;
# - definisce il percorso della cartella del soggetto;
# - stampa un riepilogo di soggetto, emisfero, seed channel,
#   file di input e directory di output;
# - restituisce:
#   json_data: configurazione aggiornata;
#   experiment_dir: directory specifica della nuova analisi;
#   sub: identificativo normalizzato del soggetto;
#   fileName: percorso base dei file EEG, senza estensione;
#   savePath: directory principale del soggetto.

# load_and_prepare_raw_data

In [ ]:
# add description of steps

raw,events,json_data=tmsu.load_and_prepare_raw_data(
    fileName=fileName,
    json_data=json_data,
    experiment_dir=experiment_dir,
    sub=sub
)

# Carica i dati EEG dal formato configurato e prepara il Raw continuo
# insieme agli eventi TMS necessari per le successive fasi della pipeline.
# La funzione:
# - richiama directorySetup() per verificare o creare la directory dell'analisi;
# - controlla che json_data contenga sourceData e dataType;
# - imposta alcuni parametri iniziali per la rimozione dell'artefatto TMS;
# - seleziona la procedura di caricamento in base al dataset indicato in sourceData;
# - per dati SIMS:
#   - carica direttamente un oggetto Epochs da file FIF;
#   - salva plot e PKL delle epoche;
#   - recupera eventi, frequenza di campionamento e statistiche inter-trial;
# - per dati MAYER in formato ASCII:
#   - legge il file ASCII con i campioni EEG e i marker;
#   - carica il file EDF associato per recuperare struttura, canali e annotazioni;
#   - uniforma i nomi dei canali;
#   - assegna i tipi corretti ai canali MK e TM;
#   - costruisce e applica il montage personalizzato dalle coordinate;
#   - sostituisce i dati EDF con i campioni letti dal file ASCII;
#   - estrae gli eventi TMS dalle annotazioni;
#   - seleziona il codice evento TMS;
#   - calcola le statistiche degli intervalli inter-trial;
#   - se use_synthetic_tep_ground_truth=True, sostituisce i segnali EEG
#     con TEP sintetici mantenendo metadati, montage, durata ed eventi originali;
# - per dati Chalfont in formato BrainVision:
#   - carica il file VHDR;
#   - assegna il montage easycap-M1;
#   - gestisce eventuali canali senza posizione;
#   - estrae e seleziona gli eventi con codice TMS 1015;
#   - calcola le statistiche inter-trial;
# - per dati UNIMI in formato BrainVision:
#   - carica il file VHDR;
#   - assegna il montage easycap-M1;
#   - estrae gli eventi con codice TMS 1128;
#   - applica, quando previsto, la correzione temporale degli eventi;
#   - calcola le statistiche inter-trial;
# - per dati UNIMI in formato ASCII:
#   - legge il file ASCII;
#   - carica l'EDF associato;
#   - rinomina i canali T3/T4/T5/T6;
#   - assegna il montage easycap-M1;
#   - sostituisce i dati EDF con quelli ASCII;
#   - estrae gli eventi TMS dalle annotazioni;
#   - calcola le statistiche inter-trial;
# - salva il layout dei sensori e i principali metadati dell'acquisizione;
# - aggiorna json_data con:
#   - frequenza di campionamento;
#   - identificativo degli eventi;
#   - nomi dei canali;
#   - statistiche inter-trial;
#   - informazioni sull'eventuale generazione sintetica;
# - restituisce:
#   raw: segnale continuo caricato e preparato;
#   events: matrice degli eventi TMS selezionati;
#   json_data: dizionario aggiornato con metadati e parametri di caricamento.

# computeBasicSteps

In [ ]:
from pathlib import Path
import importlib
import tmspath_utils as tmsu
importlib.reload(tmsu)
date,start_time=tmsu.import_modules()

raw_cleaned,epochs,detrendedEpochs,temp_epochs,json_data=tmsu.computeBasicSteps(
    raw,
    events,
    json_data,
    experiment_dir,
    sub,
    computeFOOOF=json_data["do_fooof_features"],
)

# Esegue le fasi principali di preprocessing del segnale TMS-EEG prima dell'ICA.
# La funzione:
# - riceve il segnale Raw continuo, gli eventi TMS e i parametri della pipeline;
# - se do_pulseArtifactRej=True:
#   - salva la PSD prima della correzione;
#   - rimuove o sostituisce l'artefatto immediato dell'impulso TMS;
#   - salva la PSD dopo la correzione;
#   - se do_ica_continuum=True, può eseguire una ICA preliminare sul segnale continuo;
# - se do_filter_and_plot_raw=True:
#   - applica il filtro broad-band tra l_freq e broad_band_h_freq;
#   - applica il notch alle armoniche della powerline;
#   - salva le PSD intermedie;
#   - salva il Raw filtrato in formato PKL;
# - se do_clean_trials_channels=True:
#   - crea epoche temporanee più lunghe per l'ispezione;
#   - identifica automaticamente oppure manualmente trial e canali artefattuali;
#   - elimina i trial selezionati;
#   - mantiene i canali problematici marcati come bad senza eliminarli;
#   - ricampiona le epoche temporanee a r_sfreq;
# - se do_prepare_epochs=True:
#   - crea le epoche TEP definitive usando la finestra epochs_timewindow_min/max;
#   - mantiene solo i trial selezionati durante la pulizia;
#   - seleziona i canali EEG;
#   - trasferisce la lista dei bad channels;
#   - ricampiona a r_sfreq;
#   - applica la reference media;
#   - salva plot, condition number e oggetto Epochs in formato PKL;
# - se computeFOOOF=True:
#   - esegue il FOOOF channel-wise sulle epoche pre-detrending;
# - se do_artifact=True:
#   - aggiunge un artefatto esponenziale sintetico ai canali specificati;
# - esegue computeDetrendSteps():
#   - analizza le latenze e le pendenze per stabilire se il detrending è necessario;
#   - identifica i canali offset;
#   - applica il detrending configurato;
#   - applica il notch aggiuntivo ai canali offset per i dati non simulati;
#   - calcola le metriche e i plot post-detrending;
#   - se computeFOOOF=True, esegue il FOOOF anche sulle epoche post-detrending;
#   - salva le epoche detrendate e aggiorna json_data;
# - restituisce:
#   raw_cleaned: Raw dopo rimozione impulso, filtri ed eventuale ICA sul continuo;
#   epochs: epoche TEP definitive prima del detrending;
#   detrendedEpochs: epoche dopo il detrending;
#   temp_epochs: epoche temporanee usate per selezionare trial e canali;
#   json_data: dizionario aggiornato con parametri, risultati e percorsi.

# ICA processing

In [ ]:
# se do_ica = False, la cella va runnata comunque perche la funzione gestisce entrambi True e i False restituendo epochs_after_optional_ica

(
    epochs_after_optional_ica,
    ica_model,
    json_data
)=tmsu.ICAprocessing(
    file=detrendedEpochs,
    json_data=json_data,
    experiment_dir=experiment_dir,
    sub=sub,
    computeFOOOF=json_data["do_fooof_features"]
)

# DESCRIZIONE
# Esegue la fase ICA sulle epoche detrendate.
# La funzione:
# - riceve detrendedEpochs come input;
# - verifica se json_data["do_ica"] è attivo;
# - se ICA è disattivata, restituisce direttamente una copia delle epoche detrendate,
#   imposta ica_model=None e aggiorna json_data indicando che ICA non è stata applicata;
# - se ICA è attiva:
#   - crea una cartella dedicata in 4.postICA/<timestamp>;
#   - esclude dal fit ICA i canali marcati come bad;
#   - calcola il numero di componenti ICA sui soli canali EEG validi;
#   - esegue FastICA con seed riproducibile;
#   - classifica le componenti con ICLabel;
#   - propone automaticamente componenti da escludere in base:
#       * all'etichetta ICLabel;
#       * alla probabilità minima do_label_prob_threshold;
#       * al criterio sugli autovalori do_ica_eigThresh;
#   - permette la revisione manuale delle componenti se do_ica_manualCheck=True;
#   - applica la selezione finale delle componenti alle epoche originali;
#   - mantiene nell'oggetto i canali bad, che saranno interpolati più avanti;
#   - salva i plot delle componenti incluse ed escluse;
#   - salva il modello ICA e le epoche corrette in formato PKL;
#   - aggiorna json_data con componenti totali, escluse e mantenute;
# - se computeFOOOF=True, calcola anche le feature FOOOF
#   sull'output ICA corretto;
# - restituisce:
#   epochs_after_optional_ica: epoche corrette con ICA oppure epoche detrendate se ICA è disattivata;
#   ica_model: modello ICA stimato oppure None;
#   json_data: dizionario aggiornato con risultati, parametri e percorsi.

# finalizing steps

In [ ]:
# nella versione precedente questo step era dentro a ICAprocessing, ma è meglio tenerlo separato così volendo 
# con epochs_after_optional_ica puoi vedere effetto ICA senza filtraggio, channels interpolation, nuova re-ref, etc

(
    final_epochs,
    json_data
)=tmsu.finalize_tep_epochs(
    epochs_input=epochs_after_optional_ica,
    json_data=json_data,
    experiment_dir=experiment_dir,
    sub=sub,
    computeFOOOF=json_data["do_fooof_features"],
    save_gif=True
)

# DESCRIZIONE
# 1. Verifica che l'oggetto Epochs in ingresso sia valido
# 2. Recupera il timestamp ICA oppure ne genera uno nuovo
# 3. Crea la directory di output finale in 4.postICA/<timestamp>
# 4. Registra se le epoche derivano da ICA oppure direttamente dal detrending
# 5. Esegue postICAsteps():
#    - applica il filtro FIR finale tra l_freq e h_freq
#    - ricampiona le epoche alla frequenza json_data["sfreq"]
#    - mantiene solamente i canali EEG
#    - interpola i canali marcati come bad
#    - applica la reference media
#    - salva l'oggetto postICA_final in formato PKL
#    - aggiorna i metadati relativi a filtro, sampling, canali e interpolazione
# 6. Determina la finestra temporale effettivamente disponibile per i plot
# 7. Genera i plot finali:
#    - butterfly plot del TEP medio
#    - topografie per canale
#    - PSD finale
# 8. Genera la GIF temporale butterfly + topomap e i relativi plot statici
# 9. Calcola il condition number della matrice TEP media finale
# 10. Genera le topomap finali a latenze prestabilite e la relativa animazione
# 11. Calcola la local mean field power sui seed channel
# 12. Salva il plot finale contenente il TEP medio di ciascun canale
# 13. Se do_fooof_features=True, esegue il FOOOF channel-wise sulle epoche finali
# 14. Salva una copia finale delle epoche con timestamp nella cartella 6.pkls
# 15. Aggiorna json_data con percorsi, parametri e stato della finalizzazione
# 16. Salva il file JSON aggiornato
# 17. Restituisce final_epochs e json_data

# feature extraction 

In [ ]:
if json_data['do_standard_features']:
    feature_results,json_data=tmsu.extractFeatures(
        postICA_final=final_epochs,
        json_data=json_data,
        experiment_dir=experiment_dir,
        sub=sub
    )

# DESCRIZIONE
# Se l'estrazione delle feature standard è attiva:
# - usa le epoche TEP finali, già filtrate, interpolate e referenziate;
# - calcola le feature standard sul TEP medio dei seed channel:
#   energia, integrale, sample entropy e permutation entropy;
# - se do_fooof_features=True, calcola anche offset ed esponente FOOOF
#   sul TEP medio dei seed channel;
# - calcola le matrici di correlazione tra canali e tra seed channel e tutti i canali;
# - genera e salva il plot del TEP medio dei seed channel;
# - permette l'eventuale selezione manuale dei picchi TEP;
# - calcola il PLV medio e massimo tra la fase media dei seed channel
#   e ciascun canale;
# - se do_pcist=True, calcola PCIst e le relative metriche;
# - se do_tep_natural_frequency=True, calcola la natural frequency
#   con analisi Morlet ERSP / Rosanova 2009;
# - se do_fooof_features=True, esegue anche il FOOOF channel-wise
#   sulle epoche finali;
# - salva i risultati delle feature in JSON e CSV;
# - aggiorna json_data con valori, parametri e percorsi dei file generati.

# save and load

In [ ]:
json_data=tmsu.saveLoadTestFinal(
    postICA_final=final_epochs,
    json_data=json_data,
    experiment_dir=experiment_dir,
    sub=sub,
    start_time=start_time
)

# Esegue i controlli finali, genera il joint plot conclusivo
# e registra informazioni di integrità e riproducibilità dei file salvati.
# La funzione:
# - verifica che final_epochs non sia None;
# - genera un joint plot finale del TEP:
#   - calcola il TEP medio;
#   - seleziona automaticamente fino a quattro latenze post-TMS
#     corrispondenti ai massimi della Global Field Power;
#   - impone una distanza temporale minima tra le latenze selezionate;
#   - mostra il butterfly plot insieme alle topomap alle latenze selezionate;
#   - limita il grafico alla finestra temporale configurata;
#   - salva il joint plot nella directory principale dell'esperimento;
# - aggiorna json_data con:
#   - percorso del joint plot;
#   - finestra temporale effettiva;
#   - latenze delle topomap in secondi e millisecondi;
#   - eventuali errori verificatisi durante la generazione del plot;
# - calcola gli hash SHA-256 di tutti i file PKL presenti nell'esperimento;
# - per ogni file PKL registra:
#   - hash SHA-256;
#   - dimensione in byte;
#   - data di ultima modifica;
#   - percorso assoluto;
# - calcola un hash cumulativo dell'intera collezione di file PKL;
# - calcola l'hash SHA-256 dei dati numerici contenuti in final_epochs;
# - calcola anche l'hash della selezione dei trial, quando l'input è un oggetto Epochs;
# - registra forma dei dati, numero di epoche, canali, bad channels,
#   frequenza di campionamento e limiti temporali;
# - calcola il tempo totale trascorso dall'inizio della pipeline;
# - registra il tipo dell'oggetto finale;
# - aggiorna e salva il file globale <sub>_pars.json;
# - restituisce json_data aggiornato.
